# Phase 9: Strategic Model Upgrades

This notebook analyzes the results from four strategic improvement areas:
- **9A**: Regime-Switching Ensemble (HMM/KMeans + Gated Experts)
- **9B**: Backtesting Engine with Transaction Costs
- **9C**: Model Optimization (FP16 / INT8 / ONNX)
- **9D**: Enhanced Cross-Stock Features & Retrain

**Key Finding**: After fixing a critical data leakage bug (forward-looking returns used as features), Phase 9D AUC dropped from 0.94 to a realistic 0.5226.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'figure.figsize': (10, 5)})

with open('../models/phase9_results.json') as f:
    results = json.load(f)

print('Phase 9 results loaded successfully')
print(f"Total runtime: {results['total_time_seconds']:.1f}s ({results['total_time_seconds']/60:.1f} min)")

## 9A: Regime-Switching Ensemble

We detect 3 market regimes (Bear/Sideways/Bull) using historical daily returns, then train specialist expert models for each regime. The idea is that different market conditions may need different model behaviors.

In [ ]:
r9a = results['phase9a_regime_switching']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Regime distribution
regimes = r9a['regime_counts']
colors = {'Bear': '#e74c3c', 'Sideways': '#f39c12', 'Bull': '#2ecc71'}
bars = axes[0].bar(regimes.keys(), regimes.values(), color=[colors[k] for k in regimes.keys()], edgecolor='black')
axes[0].set_title('Market Regime Distribution (Trading Days)')
axes[0].set_ylabel('Number of Days')
for bar, v in zip(bars, regimes.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10, str(v), ha='center', fontweight='bold')

# 2. Expert model validation AUC
expert_names = list(r9a['expert_val_results'].keys())
val_aucs = [r9a['expert_val_results'][n]['best_val_auc'] for n in expert_names]
bars2 = axes[1].bar(expert_names, val_aucs, color=[colors[n] for n in expert_names], edgecolor='black')
axes[1].axhline(y=0.5, color='gray', linestyle='--', label='Random (0.5)')
axes[1].set_title('Expert Validation AUC')
axes[1].set_ylabel('AUC')
axes[1].set_ylim(0.45, 0.65)
axes[1].legend()
for bar, v in zip(bars2, val_aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003, f'{v:.4f}', ha='center', fontsize=9)

# 3. Per-regime test AUC
test_names = list(r9a['per_regime_test'].keys())
test_aucs = [r9a['per_regime_test'][n]['auc'] for n in test_names]
test_ns = [r9a['per_regime_test'][n]['n'] for n in test_names]
bars3 = axes[2].bar(test_names, test_aucs, color=[colors[n] for n in test_names], edgecolor='black')
axes[2].axhline(y=0.5, color='gray', linestyle='--', label='Random (0.5)')
axes[2].axhline(y=r9a['combined_test_metrics']['auc'], color='blue', linestyle=':', label=f"Combined ({r9a['combined_test_metrics']['auc']:.4f})")
axes[2].set_title('Expert Test AUC (Out-of-Sample)')
axes[2].set_ylabel('AUC')
axes[2].set_ylim(0.45, 0.55)
axes[2].legend(fontsize=8)
for bar, v, n in zip(bars3, test_aucs, test_ns):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002, f'{v:.4f}\n(n={n})', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('../reports/phase9a_regime_ensemble.png', bbox_inches='tight')
plt.show()

print(f"\nCombined Ensemble Test AUC: {r9a['combined_test_metrics']['auc']:.4f}")
print(f"Combined Ensemble Test Acc: {r9a['combined_test_metrics']['accuracy']:.4f}")
print(f"\nConclusion: Regime-specific experts (AUC=0.498) do NOT improve over the")
print(f"unified model (AUC=0.531). Splitting data by regime reduces training samples")
print(f"per expert, increasing variance without improving bias.")

## 9B: Backtesting Engine with Transaction Costs

We backtest the model's predictions with realistic trading strategies:
- **Equal Weight**: Buy top-K predicted stocks, equal allocation
- **Confidence-Weighted**: Scale position by model confidence
- **Risk-Parity**: Inverse-volatility weighting

Each strategy tested with fee levels: 0bp, 5bp, 10bp per trade.

In [ ]:
r9b = results['phase9b_backtesting']

# Collect all strategies
strategies = {}
for k, v in r9b.items():
    if k == 'time_seconds':
        continue
    strategies[k] = v

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Returns comparison (no-fee vs with-fee)
strat_names = ['buy_and_hold', 'EqualWeight_top3_fee0.0000', 'EqualWeight_top5_fee0.0000',
               'EqualWeight_top3_fee0.0005', 'EqualWeight_top5_fee0.0005',
               'EqualWeight_top3_fee0.0010', 'EqualWeight_top5_fee0.0010']
display_names = ['Buy&Hold', 'EW3 0bp', 'EW5 0bp', 'EW3 5bp', 'EW5 5bp', 'EW3 10bp', 'EW5 10bp']
returns_pct = []
for s in strat_names:
    ret = strategies[s].get('total_return_net', 0)
    returns_pct.append(ret * 100)

colors_bar = ['#3498db'] + ['#2ecc71']*2 + ['#f39c12']*2 + ['#e74c3c']*2
bars = axes[0].bar(range(len(strat_names)), returns_pct, color=colors_bar, edgecolor='black')
axes[0].set_xticks(range(len(strat_names)))
axes[0].set_xticklabels(display_names, rotation=30, ha='right')
axes[0].set_ylabel('Total Return (%)')
axes[0].set_title('Strategy Returns: Fee Impact')
axes[0].axhline(y=0, color='black', linewidth=0.5)
for bar, v in zip(bars, returns_pct):
    if abs(v) < 200:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                     f'{v:.0f}%', ha='center', fontsize=8, fontweight='bold')

# 2. Sharpe ratio comparison
sharpes = [strategies[s].get('sharpe_ratio', 0) for s in strat_names]
bars2 = axes[1].bar(range(len(strat_names)), sharpes, color=colors_bar, edgecolor='black')
axes[1].set_xticks(range(len(strat_names)))
axes[1].set_xticklabels(display_names, rotation=30, ha='right')
axes[1].set_ylabel('Sharpe Ratio')
axes[1].set_title('Strategy Sharpe Ratios')
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].axhline(y=1.0, color='green', linestyle='--', alpha=0.5, label='Sharpe=1.0')
for bar, v in zip(bars2, sharpes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05 * np.sign(v),
                 f'{v:.2f}', ha='center', fontsize=8)
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/phase9b_backtesting.png', bbox_inches='tight')
plt.show()

print("\n" + "="*65)
print("BACKTESTING SUMMARY")
print("="*65)
print(f"{'Strategy':<30} {'Return':>10} {'Sharpe':>8} {'MaxDD':>10}")
print("-"*65)
for k, v in strategies.items():
    name = v.get('strategy', k)[:30]
    ret = v.get('total_return_net', 0) * 100
    sr = v.get('sharpe_ratio', 0)
    dd = v.get('max_drawdown', 0) * 100
    print(f"{name:<30} {ret:>9.1f}% {sr:>8.2f} {dd:>9.1f}%")

In [ ]:
# Confidence & Risk-Parity strategies
fig, ax = plt.subplots(figsize=(10, 5))

conf_strats = ['Confidence_0.52', 'Confidence_0.55', 'Confidence_0.60', 'RiskParity_top3', 'RiskParity_top5']
conf_labels = ['Conf>0.52', 'Conf>0.55', 'Conf>0.60', 'RiskPar3', 'RiskPar5']
conf_returns = [strategies[s]['total_return_net'] * 100 for s in conf_strats]
conf_sharpes = [strategies[s]['sharpe_ratio'] for s in conf_strats]

x = np.arange(len(conf_strats))
width = 0.35
bars1 = ax.bar(x - width/2, conf_returns, width, label='Return (%)', color='#e74c3c', edgecolor='black')
ax2 = ax.twinx()
bars2 = ax2.bar(x + width/2, conf_sharpes, width, label='Sharpe', color='#3498db', edgecolor='black', alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(conf_labels)
ax.set_ylabel('Total Return (%)', color='#e74c3c')
ax2.set_ylabel('Sharpe Ratio', color='#3498db')
ax.set_title('Confidence-Weighted & Risk-Parity Strategies (with 5bp fees)')
ax.axhline(y=0, color='black', linewidth=0.5)

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

print("\nKey Insight: ALL model-based strategies with realistic transaction costs (>=5bp)")
print("produce catastrophic losses. At AUC=0.53, the direction signal is too weak")
print("to overcome daily rebalancing costs. The model cannot yet be deployed profitably.")

## 9C: Model Optimization Benchmarks

Comparing inference speed and model size across precision formats:
- **FP32**: Full precision (baseline)
- **FP16**: Half precision
- **INT8**: Dynamic quantization (CPU)
- **ONNX**: Cross-platform export

In [ ]:
r9c = results['phase9c_optimization']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. Inference time comparison
formats = ['FP32', 'FP16', 'INT8']
times = [r9c['fp32']['time_ms'], r9c['fp16']['time_ms'], r9c['int8']['time_ms']]
colors_fmt = ['#3498db', '#2ecc71', '#f39c12']
bars = axes[0].bar(formats, times, color=colors_fmt, edgecolor='black')
axes[0].set_ylabel('Inference Time (ms)')
axes[0].set_title('Inference Speed (10620 samples)')
for bar, v in zip(bars, times):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{v:.0f}ms', ha='center', fontweight='bold')

# 2. Model size
sizes = [r9c['fp32']['size_mb'], r9c['fp16']['size_mb'], r9c['fp32']['size_mb'] * 0.5]  # INT8 approx
bars2 = axes[1].bar(formats, sizes, color=colors_fmt, edgecolor='black')
axes[1].set_ylabel('Size (MB)')
axes[1].set_title('Model Size')
for bar, v in zip(bars2, sizes):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{v:.2f}MB', ha='center', fontweight='bold')

# 3. Speedup factor
speedups = [1.0, r9c['fp16']['speedup'], r9c['int8']['speedup_vs_fp32_gpu']]
bars3 = axes[2].bar(formats, speedups, color=colors_fmt, edgecolor='black')
axes[2].axhline(y=1.0, color='gray', linestyle='--', label='FP32 baseline')
axes[2].set_ylabel('Speedup Factor')
axes[2].set_title('Relative Speed (>1 = faster)')
axes[2].legend()
for bar, v in zip(bars3, speedups):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{v:.2f}x', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/phase9c_optimization.png', bbox_inches='tight')
plt.show()

print(f"\nModel: {r9c['n_params']:,} parameters")
print(f"FP32: {r9c['fp32']['time_ms']:.1f}ms, {r9c['fp32']['size_mb']:.2f}MB")
print(f"FP16: {r9c['fp16']['time_ms']:.1f}ms, {r9c['fp16']['size_mb']:.2f}MB ({r9c['fp16']['speedup']:.2f}x)")
print(f"INT8: {r9c['int8']['time_ms']:.1f}ms, AUC={r9c['int8']['auc']:.4f}")
print(f"ONNX: {r9c.get('onnx', {}).get('error', 'N/A')}")
print(f"\nNote: FP16/INT8 show no speedup because the model is too small (545K params).")
print(f"Quantization overhead exceeds savings at this scale. Benefits appear at >10M params.")

## 9D: Enhanced Cross-Stock Features

We add 5 cross-stock features computed from **historical** (backward-looking) data:
1. **Sector Past Momentum**: Average 5-day past return of sector peers
2. **Market Past Momentum**: Average 5-day past return of all stocks
3. **Cross-Stock Dispersion**: Std dev of past returns across all stocks
4. **Volatility Regime**: Relative realized volatility vs cross-sectional median
5. **Sector Rank**: Percentile rank within sector by past return

### Data Leakage Fix
The initial implementation accidentally used `direction_5d_return` (forward-looking 5-day return) to compute peer momentum features. Since tech stocks move together, this perfectly leaked the target — producing an absurd AUC=0.94. After fixing to use `log_return` (historical daily return) with rolling 5-day sums, the AUC dropped to a realistic 0.5226.

In [ ]:
r9d = results['phase9d_enhanced_features']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Gate weights showing modality importance
gate_names = list(r9d['gate_weights'].keys())
gate_vals = list(r9d['gate_weights'].values())
gate_colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']
bars = axes[0].barh(gate_names, gate_vals, color=gate_colors, edgecolor='black')
axes[0].set_xlabel('Attention Gate Weight')
axes[0].set_title('Modality Importance (Enhanced Fusion)')
axes[0].set_xlim(0, 0.45)
for bar, v in zip(bars, gate_vals):
    axes[0].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{v:.3f} ({v*100:.1f}%)', va='center', fontweight='bold')

# 2. AUC comparison: baseline vs enhanced
models = ['Baseline\n(Clean Fusion)', 'Enhanced\n(+Cross-Stock)', 'Leaky Version\n(INVALID)']
aucs = [0.5312, r9d['test_metrics']['auc'], 0.9390]
model_colors = ['#3498db', '#2ecc71', '#e74c3c']
bars2 = axes[1].bar(models, aucs, color=model_colors, edgecolor='black')
axes[1].axhline(y=0.5, color='gray', linestyle='--', label='Random (0.5)')
axes[1].set_ylabel('Test AUC')
axes[1].set_title('AUC: Enhanced Features (Fixed vs Leaky)')
axes[1].set_ylim(0.4, 1.0)
for bar, v in zip(bars2, aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{v:.4f}', ha='center', fontweight='bold')
# Add red X over leaky version
axes[1].text(2, 0.85, '\u2717', fontsize=40, color='red', ha='center', va='center', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/phase9d_enhanced_features.png', bbox_inches='tight')
plt.show()

print(f"\nEnhanced Model:")
print(f"  Parameters: {r9d['n_params']:,} (vs 545,671 baseline)")
print(f"  Val AUC:  {r9d['best_val_auc']:.4f}")
print(f"  Test AUC: {r9d['test_metrics']['auc']:.4f}")
print(f"  Test Acc: {r9d['test_metrics']['accuracy']:.4f}")
print(f"\n  Gate weight for enhanced features: {r9d['gate_weights']['enhanced']:.3f} (22.5%)")
print(f"  Price remains dominant: {r9d['gate_weights']['price']:.3f} (34.2%)")
print(f"\nConclusion: Cross-stock features from PAST returns add marginal signal")
print(f"(AUC 0.523 vs 0.531 baseline). The model correctly learns to weight them")
print(f"at 22.5%, but they slightly hurt test performance — a common outcome when")
print(f"adding noisy features to an already-trained model.")

## Overall Summary & Conclusions

In [ ]:
# Final summary figure
fig, ax = plt.subplots(figsize=(12, 6))

phases = [
    'Phase 4\nPrice CNN-BiLSTM',
    'Phase 5\nGAT + Graph',
    'Phase 6\nGated Fusion',
    'Phase 7\nAblations',
    'Phase 8\nV2 Clean Fusion',
    'Phase 8 CF\n(Filtered 6.3%)',
    'Phase 9A\nRegime Ensemble',
    'Phase 9D\nEnhanced Features',
]
aucs = [0.5109, 0.5247, 0.5192, 0.5149, 0.5312, 0.6024, 0.4976, 0.5226]
phase_colors = ['#bdc3c7']*5 + ['#f39c12'] + ['#e74c3c', '#3498db']

bars = ax.bar(range(len(phases)), aucs, color=phase_colors, edgecolor='black', width=0.7)
ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1, label='Random Baseline (0.5)')
ax.set_xticks(range(len(phases)))
ax.set_xticklabels(phases, fontsize=8)
ax.set_ylabel('Test AUC', fontsize=12)
ax.set_title('Model Evolution: All Phases', fontsize=14, fontweight='bold')
ax.set_ylim(0.45, 0.65)
ax.legend(fontsize=10)

for bar, v in zip(bars, aucs):
    color = 'green' if v > 0.5 else 'red'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{v:.4f}', ha='center', fontsize=9, fontweight='bold', color=color)

plt.tight_layout()
plt.savefig('../reports/phase9_overall_summary.png', bbox_inches='tight')
plt.show()

print("="*70)
print("PHASE 9 STRATEGIC UPGRADES - FINAL SUMMARY")
print("="*70)
print()
print("Phase 9A (Regime Switching):")
print(f"  Combined AUC: 0.4976 (-3.4% vs baseline)")
print(f"  Verdict: NEGATIVE. Splitting data reduces sample size per expert,")
print(f"  increasing variance. Regime detection itself is noisy.")
print()
print("Phase 9B (Backtesting):")
print(f"  Best zero-fee: EW5 Sharpe=1.66, Return=+1263%")
print(f"  With 5bp fees: All strategies deeply unprofitable")
print(f"  Verdict: Model signal too weak for profitable trading.")
print(f"  Need AUC>0.55+ consistently to overcome transaction costs.")
print()
print("Phase 9C (Optimization):")
print(f"  FP16: 50% size reduction, ~same speed (model too small to benefit)")
print(f"  INT8: Preserves AUC=0.525, no speed gain at 545K params")
print(f"  Verdict: Optimization matters at scale (>10M params).")
print()
print("Phase 9D (Enhanced Features):")
print(f"  AUC: 0.5226 (-0.9% vs baseline)")
print(f"  Verdict: Historical cross-stock momentum adds noise, not signal.")
print(f"  Gate correctly weights them at 22.5% but they hurt test performance.")
print()
print("OVERALL: Best full-coverage model remains Phase 8 V2 Clean Fusion (AUC=0.5312).")
print("Best filtered model: Confidence>0.6 (AUC=0.6024, 6.3% coverage).")
print("\nThe fundamental challenge remains: 5-day equity direction prediction")
print("from public data is near-efficient-market-hard. Our AUC=0.53 represents")
print("a genuine but very small edge that transaction costs currently erase.")